<a href="https://colab.research.google.com/github/msrehman786/IBM-AI-Certification/blob/main/Voice_Assistant_With_OpenAI's_GPT_Model_and_IBM_Watson.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/ibm-developer-skills-network/bkrva-chatapp-with-voice-and-openai-outline.git
!mv bkrva-chatapp-with-voice-and-openai-outline chatapp-with-voice-and-openai-outline
!cd chatapp-with-voice-and-openai-outline

Cloning into 'bkrva-chatapp-with-voice-and-openai-outline'...
remote: Enumerating objects: 36, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 36 (delta 1), reused 1 (delta 1), pack-reused 29 (from 1)
Receiving objects: 100% (36/36), 19.32 KiB | 760.00 KiB/s, done.
Resolving deltas: 100% (6/6), done.


In [2]:
!python3 -m pip install flask
!python3 -m pip install flask_cors

Integrating your chatbot into your Flask server

In [3]:
#!python3 -m pip install transformers==4.41.2
#!python3 -m pip install torch==2.11.0
#!python3 -m pip install accelerate==0.30.1
#!python3 -m pip install numpy==1.26.4

!pip install transformers pillow torch torchvision torchaudio accelerate numpy

#!python3 -m pip install transformers
#!python3 -m pip install torch
#!python3 -m pip install accelerate
#!python3 -m pip install numpy


copy the code to initialize your chatbot

In [4]:
!pip install flask pyngrok flask_cors

In [5]:
!which ngrok

/usr/local/bin/ngrok


In [6]:
!ngrok authtoken '3IBE5FhsNRF52VxuQwh0JnlsAqu_6RoqvbJKa7EwEAespyiP'

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [7]:
!python3 -m pip install -r requirements.txt

In [15]:
!mkdir chatapp-with-voice-and-openai-outline/certs/
!cp /usr/local/share/ca-certificates/rootCA.crt chatapp-with-voice-and-openai-outline/certs/

cp: cannot stat '/usr/local/share/ca-certificates/rootCA.crt': No such file or directory


worker.py

In [22]:
from openai import OpenAI
import requests
from google.colab import userdata

# Retrieve the API key from Colab secrets
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

# Pass the API key to the OpenAI client
openai_client = OpenAI(api_key=OPENAI_API_KEY)


def speech_to_text(audio_binary):

	# Set up Watson Speech-to-Text HTTP Api url
	base_url = "https://sn-watson-stt.labs.skills.network"
	api_url = base_url+'/speech-to-text/api/v1/recognize'

	# Set up parameters for our HTTP request
	params = {
		'model': 'en-US_Multimedia',
	}

	# Set up the body of our HTTP request
	body = audio_binary

	# Send a HTTP Post request
	response = requests.post(api_url, params=params, data=audio_binary).json()

	# Parse the response to get our transcribed text
	text = 'null'
	while bool(response.get('results')):
		print('speech to text response:', response)
		text = response.get('results').pop().get('alternatives').pop().get('transcript')
		print('recognised text: ', text)
		return text


def text_to_speech(text, voice=""):

	# Set up Watson Text-to-Speech HTTP Api url
	base_url = 'https://sn-watson-tts.labs.skills.network'
	api_url = base_url + '/text-to-speech/api/v1/synthesize?output=output_text.wav'

	# Adding voice parameter in api_url if the user has selected a preferred voice
	if voice != "" and voice != "default":
		api_url += "&voice=" + voice

	# Set the headers for our HTTP request
	headers = {
		'Accept': 'audio/wav',
		'Content-Type': 'application/json',
	}

	# Set the body of our HTTP request
	json_data = {
		'text': text,
	}

	# Send a HTTP Post request to Watson Text-to-Speech Service
	response = requests.post(api_url, headers=headers, json=json_data)
	print('text to speech response:', response)
	return response.content


def openai_process_message(user_message):
    # Set the prompt for OpenAI Api
    prompt = "Act like a personal assistant. You can respond to questions, translate sentences, summarize news, and give recommendations. Keep responses concise - 2 to 3 sentences maximum."
    # Call the OpenAI Api to process our prompt
    openai_response = openai_client.chat.completions.create(
        model="gpt-5-nano",
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": user_message}
        ],
        max_completion_tokens=1000
    )
    print("openai response:", openai_response)
    # Parse the response to get the response message for our prompt
    response_text = openai_response.choices[0].message.content
    return response_text

server.py

In [27]:
import base64
import json
from flask import Flask, render_template, request
from worker import speech_to_text, text_to_speech, openai_process_message
from flask_cors import CORS
import os
from pyngrok import ngrok

app = Flask(__name__)
cors = CORS(app, resources={r"/*": {"origins": "*"}})

# Optional: Set ngrok auth token (skip if not using one)
ngrok.set_auth_token("3IBE5FhsNRF52VxuQwh0JnlsAqu_6RoqvbJKa7EwEAespyiP")  # Replace with your token

# Open a ngrok tunnel to port 5000 (where Flask will run)
public_url = ngrok.connect(5001).public_url
print(f"✅ Flask app is live at: {public_url}")

@app.route('/', methods=['GET'])
def index():
    return render_template('index.html')


@app.route('/speech-to-text', methods=['POST'])
def speech_to_text_route():
    print("processing speech-to-text")
    audio_binary = request.data # Get the user's speech from their request
    text = speech_to_text(audio_binary) # Call speech_to_text function to transcribe the speech

	# Return the response back to the user in JSON format
    response = app.response_class(
        response=json.dumps({'text': text}),
        status=200,
        mimetype='application/json'
    )
    print(response)
    print(response.data)
    return response


@app.route('/process-message', methods=['POST'])
def process_message_route():
    user_message = request.json['userMessage'] # Get user's message from their request
    print('user_message', user_message)

    voice = request.json['voice'] # Get user's preferred voice from their request
    print('voice', voice)

	# Call openai_process_message function to process the user's message and get a response back
    openai_response_text = openai_process_message(user_message)

	# Clean the response to remove any emptylines
    openai_response_text = os.linesep.join([s for s in openai_response_text.splitlines() if s])

	# Call our text_to_speech function to convert OpenAI Api's reponse to speech
    openai_response_speech = text_to_speech(openai_response_text, voice)

    # convert openai_response_speech to base64 string so it can be sent back in the JSON response
    openai_response_speech = base64.b64encode(openai_response_speech).decode('utf-8')

	# Send a JSON response back to the user containing their message's response both in text and speech formats
    response = app.response_class(
        response=json.dumps({"openaiResponseText": openai_response_text, "openaiResponseSpeech": openai_response_speech}),
        status=200,
        mimetype='application/json'
    )

    print(response)
    return response


if __name__ == "__main__":
    #app.run(port=8000, host='0.0.0.0')
    app.run(host='0.0.0.0', port=5001)


✅ Flask app is live at: https://snowless-persuader-nest.ngrok-free.dev
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5001
 * Running on http://172.28.0.12:5001
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [02/Sep/2026 12:03:07] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [02/Sep/2026 12:03:07] "GET /static/style.css HTTP/1.1" 304 -
INFO:werkzeug:127.0.0.1 - - [02/Sep/2026 12:03:07] "GET /static/script.js HTTP/1.1" 304 -


user_message hello
voice default
openai response: ChatCompletion(id='chatcmpl-EJdyWHSRUGQjuE2kfl9g1IZkeWinx', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello! How can I assist you today? I can translate, summarize news, or offer recommendations—tell me what you need.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1788350604, model='gpt-5-nano-2025-08-07', object='chat.completion', moderation=None, service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=419, prompt_tokens=45, total_tokens=464, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=384, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0)))
